# Hand Cropping Demo 
### For CNN-LSTM Hybrid Model

In [ ]:
!pip install mediapipe==0.10.14 matplotlib opencv-python

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import cv2
import torch
import numpy as np
import mediapipe as mp
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
MODEL_DIR = '/content/drive/MyDrive/LearningASL_models'
VIDEO_ROOT = '/content/hf_asl_videos'
ZIP_PATH = '/content/drive/MyDrive/hf_asl_videos.zip'

if not os.path.exists(VIDEO_ROOT):
    !unzip -q {ZIP_PATH} -d /content/

words = sorted([d for d in os.listdir(VIDEO_ROOT) if os.path.isdir(os.path.join(VIDEO_ROOT, d))])
print(f'Loaded {len(words)} classes: {words[:5]}...')

In [ ]:
class LSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim*2, num_classes)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

# load CNN
cnn = resnet18(weights=None)
cnn.fc = nn.Linear(cnn.fc.in_features, len(words)) # init with classification head
state_dict = torch.load(os.path.join(MODEL_DIR, 'best_hand_cnn.pth'), map_location=device)

try:
    cnn.load_state_dict(state_dict)
except RuntimeError:
    print("Standard load failed, trying with Identity head")
    cnn.fc = nn.Identity()
    cnn.load_state_dict(state_dict)

# use Identity head for feature extraction
if not isinstance(cnn.fc, nn.Identity):
    cnn.fc = nn.Identity()

cnn = cnn.to(device).eval()

# load LSTM
lstm = LSTM(512, 128, len(words))
lstm.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'best_hand_lstm.pth'), map_location=device))
lstm = lstm.to(device).eval()

In [ ]:
mp_hands = mp.solutions.hands
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def crop_hand(frame, landmarks, padding=20):
    h, w, _ = frame.shape
    x_min, y_min = w, h
    x_max, y_max = 0, 0
    for lm in landmarks.landmark:
        x, y = int(lm.x * w), int(lm.y * h)\n        x_min, y_min = min(x_min, x), min(y_min, y)
        x_max, y_max = max(x_max, x), max(y_max, y)

    x_min, y_min = max(0, x_min-padding), max(0, y_min-padding)
    x_max, y_max = min(w, x_max+padding), min(h, y_max+padding)
    return frame[y_min:y_max, x_min:x_max]

def predict_video(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret: break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()

    if len(frames) < 10: return None, None, None

    indices = np.linspace(0, len(frames)-1, 30, dtype=int)
    cnn_features = []
    crops_to_show = []

    with mp_hands.Hands(static_image_mode=True, max_num_hands=1) as hands:
        for idx in indices:
            frame = frames[idx]
            results = hands.process(frame)

            crop = np.zeros((128, 128, 3), dtype=np.uint8)
            if results.multi_hand_landmarks:
                c = crop_hand(frame, results.multi_hand_landmarks[0])
                if c.size > 0:
                    crop = cv2.resize(c, (128, 128))

            crops_to_show.append(crop)
            tensor = transform(Image.fromarray(crop)).unsqueeze(0).to(device)
            with torch.no_grad():
                cnn_features.append(cnn(tensor).cpu().numpy())

    # LSTM prediction
    seq = torch.tensor(np.vstack(cnn_features)).float().unsqueeze(0).to(device)
    with torch.no_grad():
        logits = lstm(seq)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

    display_crop = crops_to_show[len(crops_to_show)//2]
    for c in crops_to_show:
        if c.sum() > 0:
            display_crop = c
            break

    return frames[len(frames)//2], display_crop, probs

## Visualize Crop Predictions

In [ ]:
def visualize_prediction(true_label):
    # random video of true_label class
    class_dir = os.path.join(VIDEO_ROOT, true_label)
    videos = [f for f in os.listdir(class_dir) if f.endswith('.mp4')]
    if not videos: return

    vid_path = os.path.join(class_dir, np.random.choice(videos))

    orig_frame, hand_crop, probs = predict_video(vid_path)
    if probs is None:
        print("Could not process video")
        return

    # top 3
    top3_idx = probs.argsort()[-3:][::-1]
    top3_words = [words[i] for i in top3_idx]
    top3_probs = probs[top3_idx]

    # plot
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))

    ax[0].imshow(orig_frame)
    ax[0].set_title(f"Original Video: '{true_label}'")
    ax[0].axis('off')

    ax[1].imshow(hand_crop)
    ax[1].set_title("CNN Input (Cropped Hand)")
    ax[1].axis('off')

    y_pos = np.arange(3)
    colors = ['green' if w == true_label else 'gray' for w in top3_words]
    ax[2].barh(y_pos, top3_probs, align='center', color=colors)
    ax[2].set_yticks(y_pos)
    ax[2].set_yticklabels(top3_words)
    ax[2].invert_yaxis()
    ax[2].set_xlabel('Confidence')
    ax[2].set_title('Model Prediction')
    ax[2].set_xlim(0, 1.0)

    plt.tight_layout()
    plt.show()

In [ ]:
# demo 3 random words
test_words = np.random.choice(words, 3, replace=False)

for w in test_words:
    visualize_prediction(w)